In [0]:
# ============================================================
# GOLD LAYER - DIM_CUSTOMER - UPDATED FOR CI/CD
# ============================================================

import os
from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# LOAD CONFIGURATION FROM ENVIRONMENT VARIABLES
# ============================================================

# Get environment (DEV, TEST, PROD) - Default to DEV
try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from widget: {env}")
except:
    env = os.getenv('ENV', 'DEV')
    print(f"📌 Environment from env var: {env}")

# Get storage account and access key from environment (GitHub Secrets)
storage_account_name = os.getenv('STORAGE_ACCOUNT', 'capstonestorage01')
access_key = os.getenv('STORAGE_ACCESS_KEY')

# Set container names based on environment
if env == 'DEV':
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    silver_container = 'silver'
    gold_container = 'gold'

# Configure Spark with access key (from GitHub Secrets)
if access_key:
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        access_key
    )
    print("✅ Storage access key configured from GitHub Secrets")
else:
    print("⚠️ WARNING: No access key found! Using Azure AD authentication")

# Dynamic paths based on environment
SILVER = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
GOLD   = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 SILVER Container: {silver_container}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 SILVER Path: {SILVER}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

# ============================================================
# YOUR EXISTING CODE (NO CHANGES NEEDED BELOW!)
# ============================================================

# ── Load ──
df_customer = spark.read.format("delta").load(f"{SILVER}/customer_master")

# ── Transform ──
dim_customer = df_customer.select(
    F.col("customer_id"),
    F.col("age"),
    F.col("gender"),
    F.col("city"),
    F.col("employment_type"),
    F.col("annual_income"),
    F.col("risk_segment"),
    F.when(F.col("annual_income") < 500000,  F.lit("Low Income"))
     .when(F.col("annual_income") < 1000000, F.lit("Mid Income"))
     .otherwise(F.lit("High Income"))
     .alias("income_band"),
    F.when(F.col("age") < 30, F.lit("18-29"))
     .when(F.col("age") < 45, F.lit("30-44"))
     .when(F.col("age") < 60, F.lit("45-59"))
     .otherwise(F.lit("60+"))
     .alias("age_group"),
    F.current_timestamp().alias("gold_created_at")
)

# ── Write ──
dim_customer.write.format("delta").mode("overwrite")\
    .save(f"{GOLD}/dim_customer")

print(f"\n{'='*55}")
print(f"✅ dim_customer : {dim_customer.count():,} rows")
print(f"📁 Written to: {GOLD}/dim_customer")
print(f"🏭 Environment: {env}")
print(f"{'='*55}")

✅ dim_customer : 100,000 rows
